# Appendix R — Majority: a structurally distinct handover task

Reproduces **Appendix R** of *Same Behaviour, Different Causal Route* (Tables R1–R6, Figure R1) and the Majority sentences in §5.5, §7 and §8.

**Task.** `A C B C D C … [READOUT] → C`. There are `L` symbols from an 8-symbol alphabet in random order, rejection-sampled to have a unique mode (margin ≥ 1), followed by 3 delay tokens and `[READOUT]`. The target is the modal symbol (chance 1/8). No single position is informative, so the task cannot be solved by a one-position pointer.

**Instrument.** The model, training loop, price, resampling dependence (`Dh`, `Dc`) and switch mechanics are copied verbatim from `experiments/core/symbolic_main.ipynb`. `[READOUT]` reuses the `CUE` token id, so the hybrid has the same 33,028 parameters. New in this notebook: the task, route **deletion** (`logits = l_h`), and attention-structure statistics.

## Protocol and disclosure
| Section | What | Decides | Paper |
|---|---|---|---|
| 1. Screen | free hybrid and attention-free GRU over the **pre-declared** `LOAD_GRID` | GO / NO-GO, `L_pos` | Table R1 |
| 2. Price | priced-from-init dose–response at `L_pos`; λ = 3·λ\* (App. C rule) | λ | §R.1 |
| 3. Switch timing | unpriced pilot; switch once solved and retrieval-dominant (App. B rule) | switch step | §R.1 |
| 4. Four-arm | never / late / maintained / withdrawn at `L_pos`, deletion at every endpoint | handover | Table R2, Fig. R1a–d |
| 5. Controls | matched post-hoc attenuation, attention structure, length shift | alternatives | Tables R3–R5, Fig. R1e–g |
| 6. Boundary search | length extension (`EXTRA_LOADS`), then alphabet size (`K_GRID`) | `L_neg` / `K_neg` | Tables R1†, R6 |
| 7. Tables, figure, summary | built only from stored results | — | Appendix R |

**Thresholds** (cell below) were declared before any steering outcome. `L_pos` is the highest load in `LOAD_GRID` that passes all three screen criteria. The pre-declared grid produced no non-viable control. The length extension (`EXTRA_LOADS`) and then the alphabet axis (`K_GRID`) were added afterwards, in that order. Both are selected **only** from pre-intervention viability, and neither produced a control. Selections are written to `selection__majority.json` before the endpoints that depend on them, and are never re-decided on resume.

## Running
- **Regenerate tables and figure without training:** set `WORKDIR` to a folder containing the released `results_majority/`, keep `RESUME=True`, and run all cells. Every stored unit is skipped. The two controls that evaluate saved models, and the Section 6 phases that need checkpoints, run only when `checkpoints_majority/` is present; the tables use the stored values.
- **Full reproduction:** `PRESET="full"`, about 2 GPU-hours on one T4. Run `PRESET="smoke"` first as a pipeline check (about 5 min; its numbers are meaningless).
- Every unit of work is appended to JSONL stores and resumable. Seeds: screen 3, endpoints 5.

In [ ]:
# Requirements: torch, numpy, pandas, matplotlib (tested with Python 3.12, PyTorch 2.x)
import copy, hashlib, itertools, json, math, os, pathlib, platform, random, time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== CHOOSE HERE ==============================
PRESET  = "full"     # "smoke" (pipeline check, ~5 min) | "quick" | "full" (paper numbers)
RESUME  = True       # skip every unit of work already stored on disk
WORKDIR = os.environ.get("PAPER_WORKDIR", ".")
# To regenerate the paper tables/figure WITHOUT retraining, point WORKDIR at a
# folder containing results_majority/ (e.g. the released results) and keep RESUME.

RUN = {"screen": True, "price": True, "four_arm": True,
       "posthoc": True, "posthoc_curve": True, "structure": True, "length_shift": True,
       "availability": False,                       # optional control, not reported
       "boundary_length": True, "boundary_alphabet": True,
       "tables": True, "figure": True, "summary": True}
# =========================================================================

MODALITY, IS_VISUAL = "abstract", False
TAG = "majority"

# ---- pre-declared protocol (fixed before any steering outcome was observed) ----
N_SYMBOLS    = 8          # alphabet = output classes (chance 1/8)
MARGIN       = 1          # modal count minus runner-up count (1 = strict unique mode)
LOAD_GRID    = [4, 6, 8, 12, 16, 24, 32]   # pre-declared screen; L_pos is chosen here
VIABLE_ACC   = 0.95       # alternative-route viability screen (paper Sec. 5.2)
ACC_FLOOR    = 0.90       # validity floor for crossings / pre-switch convergence
DC_MIN, DH_MAX = 0.80, 0.20   # "retrieval-dominant"
DELETION_MIN = 0.50       # normalised accuracy lost when retrieval is deleted
LAM_MULT     = 3.0        # four-arm price = 3 x lambda*(L_pos)   (App. C rule)
LAM_GRID     = [0.003, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35]
DEL_SAFE     = 0.01       # "retrieval deletable": deletion costs <= 1 point of accuracy
SCREEN_GRID  = LOAD_GRID

# ---- boundary search (declared after the pre-declared screen found no control) ----
EXTRA_LOADS  = [48, 64, 96, 128]   # length axis, above L_pos
K_GRID       = [16, 32, 64]        # alphabet axis at L = L_pos (declared after EXTRA_LOADS)
NEG_ARMS     = ["never", "late", "maintained"]
POSTHOC_REF_LAMS = [0.003, 0.005, 0.01, 0.02]   # non-zero-share attenuation references

_sfx = "" if PRESET == "full" else f"_{PRESET}"
WORK = pathlib.Path(WORKDIR).expanduser().resolve()
RESDIR  = WORK / f"results_majority{_sfx}"
CKPTDIR = WORK / f"checkpoints_majority{_sfx}"
FIGDIR  = WORK / f"figures_majority{_sfx}"
TABDIR  = RESDIR / "tables"
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)


def S(full, quick=None, smoke=None):
    """Budget for the current preset."""
    return {"full": full,
            "quick": quick if quick is not None else max(full // 3, 100),
            "smoke": smoke if smoke is not None else max(full // 40, 100)}[PRESET]


STEPS    = S(2500, 1200, 150)   # default per-run budget (cosine), as in the paper
REC_LONG = 3 * STEPS            # recurrence-alone viability budget (7500)
PILOT    = S(8000, 3000, 300)   # unpriced pilot used only to time the switch
SW_MIN   = S(2000, 800, 100)    # earliest allowed switch (paper: step 2000)
POST     = S(10000, 3000, 300)  # steps after the switch
LOG      = S(500, 250, 50)      # logging interval for trajectories

SEEDS_SCREEN = {"full": [0, 1, 2], "quick": [0, 1], "smoke": [0]}[PRESET]
SEEDS_END    = {"full": [0, 1, 2, 3, 4], "quick": [0, 1], "smoke": [0]}[PRESET]
FORCE_CONTINUE = PRESET == "smoke"   # a smoke run must exercise every cell

print(f"device {DEVICE} | torch {torch.__version__} | python {platform.python_version()}")
print(f"preset {PRESET} | resume {RESUME} | results -> {RESDIR}")
if PRESET == "smoke":
    print("SMOKE preset: numbers are meaningless; this only checks the pipeline.")


def csv(name):
    return str(RESDIR / f"{name}__{TAG}.csv")


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)

## Persistence (verbatim from the core suite) and the frozen selection record

In [ ]:
def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, pathlib.Path):
        return str(o)
    return str(o)


def keyof(spec):
    return hashlib.md5(
        json.dumps(spec, sort_keys=True, default=_jsonable).encode()).hexdigest()[:16]


def cfg_tag(cfg=None):
    if cfg is None:
        return "default"
    return ",".join(f"{k}={getattr(cfg, k)}" for k in sorted(cfg.__dataclass_fields__))


class Store:
    """Append-only JSONL of finished units of work, one file per block."""

    def __init__(self, block):
        self.block = block
        self.path = RESDIR / f"{block}__{TAG}.jsonl"
        self.recs, bad = {}, 0
        if RESUME and self.path.exists():
            for line in self.path.read_text().splitlines():
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                    self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    bad += 1
            if bad:
                print(f"  [{block}] discarded {bad} unreadable record(s)")

    def has(self, spec):
        return RESUME and keyof(spec) in self.recs

    def get(self, spec):
        return self.recs[keyof(spec)]

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=_jsonable) + "\n")
            f.flush(); os.fsync(f.fileno())
        return r

    def frame(self):
        out = []
        for r in self.recs.values():
            spec = dict(r["_spec"])
            body = {k: v for k, v in r.items()
                    if not k.startswith("_") and k not in ("trace", "rows")}
            if "rows" in r:
                for row in r["rows"]:
                    out.append({**spec, **body, **row})
            else:
                out.append({**spec, **body})
        return pd.DataFrame(out)

    def traces(self):
        out = []
        for r in self.recs.values():
            for t in r.get("trace", []):
                out.append({**r["_spec"], **t})
        return pd.DataFrame(out)


def cached(store, spec, fn, label=None):
    if store.has(spec):
        return store.get(spec)
    t0 = time.time()
    payload = fn()
    payload["_elapsed"] = time.time() - t0
    r = store.put(spec, payload)
    if label:
        print(f"  [{store.block}] {label}  {payload['_elapsed']/60:.1f}m")
    return r


def ckpt_path(name):
    return CKPTDIR / f"{name}__{TAG}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    torch.save(obj, str(p) + ".tmp")
    os.replace(str(p) + ".tmp", str(p))


def ckpt_load(name):
    p = ckpt_path(name)
    if not (RESUME and p.exists()):
        return None
    try:
        return torch.load(str(p), map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(str(p), map_location=DEVICE)
    except Exception as e:
        print(f"  checkpoint {p.name} unreadable ({e}) — recomputing")
        return None


def save_model(name, m):
    ckpt_save(name, {"state": m.state_dict()})


def load_model_into(name, m):
    ck = ckpt_load(name)
    if ck is None:
        return False
    m.load_state_dict(ck["state"])
    return True


# ---- the selection record: written once, never re-decided ----------------
SEL_PATH = RESDIR / f"selection__{TAG}.json"


def load_sel():
    return json.loads(SEL_PATH.read_text()) if SEL_PATH.exists() else {}


def save_sel(sel):
    SEL_PATH.write_text(json.dumps(sel, indent=2, default=_jsonable))


def go_flag(sel):
    return bool(sel.get("go")) or FORCE_CONTINUE


print(f"persistence: {RESDIR}\nselection record: {SEL_PATH} "
      f"({'exists' if SEL_PATH.exists() else 'not yet written'})")

## Configuration and tasks

`SelectiveRecall` is copied unchanged and used only as the reference for the attention-structure comparison. `Majority` is new.

In [ ]:
PAD, DELAY, CUE, MARK = 0, 1, 2, 3
N_SPECIAL = 4
MAX_POS = 24
D_FEAT = 64
READOUT = CUE   # [READOUT] reuses the CUE id: same vocabulary, same embedding
                # table, same 33,028-parameter hybrid as the paper


@dataclass
class Config:
    n_symbols: int = N_SYMBOLS
    max_query_pos: int = MAX_POS
    base_delay: int = 3
    d_model: int = 64
    lr: float = 3e-3
    batch_size: int = 128
    weight_decay: float = 0.0
    grad_clip: float = 1.0


CFG = Config()
N_CLASSES = CFG.n_symbols
IN_DIM = D_FEAT + MAX_POS + 3


def vocab_size(cfg):
    return N_SPECIAL + cfg.n_symbols + cfg.max_query_pos


def token_str(t, cfg=CFG):
    t = int(t)
    return {PAD: "_", DELAY: ".", CUE: "R", MARK: "*"}.get(
        t, chr(ord("A") + t - N_SPECIAL) if t < N_SPECIAL + cfg.n_symbols
        else f"Q{t - N_SPECIAL - cfg.n_symbols}")


class Task:
    name = "base"

    def __init__(self, cfg=None, split="train"):
        self.cfg = cfg or CFG
        self.split = split

    @property
    def n_out(self):
        return self.cfg.n_symbols

    def show(self, load, n=3):
        b = self.generate(n, load, device=DEVICE)
        for i in range(n):
            print("  " + " ".join(token_str(t) for t in b["x"][i].cpu())
                  + f"  →  {token_str(int(b['y'][i]) + N_SPECIAL)}")


class SelectiveRecall(Task):
    """Verbatim from the core suite (abstract branch)."""
    name = "selective_recall"

    def generate(self, B, load, extra_delay=0, device=DEVICE, extra_items=0):
        c, n = self.cfg, load + extra_items
        assert n <= c.max_query_pos
        lab = torch.randint(0, c.n_symbols, (B, n), device=device)
        j = torch.randint(0, load, (B,), device=device)
        ar = torch.arange(B, device=device)
        nd = c.base_delay + extra_delay
        delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
        q = (j + N_SPECIAL + c.n_symbols).unsqueeze(1)
        x = torch.cat([lab + N_SPECIAL, delay, q], 1)
        return {"x": x, "y": lab[ar, j], "key_pos": j, "items": lab}


class Majority(Task):
    """items (random order) · DELAY x base_delay · [READOUT]  ->  the unique modal symbol.

    Rows are rejection-sampled from i.i.d. uniform symbols, keeping only rows
    whose modal count exceeds the runner-up count by at least `margin`. The
    modal symbol is therefore uniform over the alphabet and its occurrences are
    at uniformly random positions: no single position is informative, so a
    one-position pointer cannot solve the task. `load` is the number of items.
    """
    name = "majority"

    def __init__(self, cfg=None, split="train", margin=None):
        super().__init__(cfg, split)
        self.margin = MARGIN if margin is None else margin

    def _rows(self, B, L, device):
        K, parts, have, tries = self.cfg.n_symbols, [], 0, 0
        while have < B:
            n = max(2 * (B - have), 256)
            s = torch.randint(0, K, (n, L), device=device)
            cnt = F.one_hot(s, K).sum(1)
            top = cnt.topk(2, dim=1).values
            ok = (top[:, 0] - top[:, 1]) >= self.margin
            if ok.any():
                parts.append(s[ok][:B - have])
                have += parts[-1].shape[0]
            tries += 1
            if tries > 1000:
                raise RuntimeError(f"majority: acceptance too low at L={L}, "
                                   f"margin={self.margin}")
        return torch.cat(parts, 0)

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        lab = self._rows(B, load, device)                     # B, L
        cnt = F.one_hot(lab, c.n_symbols).sum(1)              # B, K
        y = cnt.argmax(1)
        second = cnt.scatter(1, y.unsqueeze(1), -1).max(1).values
        runner = (cnt == second.unsqueeze(1)) & (second.unsqueeze(1) > 0)
        nd = c.base_delay + extra_delay
        delay = torch.full((B, nd), DELAY, device=device, dtype=torch.long)
        ro = torch.full((B, 1), READOUT, device=device, dtype=torch.long)
        x = torch.cat([lab + N_SPECIAL, delay, ro], 1)
        return {"x": x, "y": y, "items": lab, "counts": cnt, "runner_up": runner}


TASKS = {t.name: t for t in [SelectiveRecall, Majority]}


def make_task(name="majority", cfg=None, split="train", **kw):
    return TASKS[name](cfg, split, **kw)


print("majority, load 8:")
make_task("majority").show(8, 3)

# sanity: the target is the unique mode, and the mode is uniform over symbols
_b = make_task("majority").generate(4096, 8)
assert (_b["counts"].max(1).values == _b["counts"].gather(1, _b["y"][:, None]).squeeze(1)).all()
assert ((_b["counts"] == _b["counts"].max(1, keepdim=True).values).sum(1) == 1).all()
_freq = torch.bincount(_b["y"], minlength=N_SYMBOLS).float() / 4096
print(f"\nmode frequency per symbol (should be ~{1/N_SYMBOLS:.3f}): "
      f"{np.round(_freq.cpu().numpy(), 3)}")
for L in LOAD_GRID:
    _c = make_task("majority").generate(2048, L)["counts"].topk(2, 1).values
    print(f"  L={L:>2}: mean modal count {_c[:,0].float().mean():.2f}, "
          f"mean runner-up {_c[:,1].float().mean():.2f}")

## Model, training, dependence (verbatim) + deletion and attention structure (new)

Deletion removes a route's logits (`logits = l_h`), matching the large-scale notebook's `drop="retrieval"`. It is the necessity test that resampling cannot replace (App. F, Q).

In [ ]:
def ablate(v, spec):
    if v is None or spec is None:
        return v
    mode, strength = spec if isinstance(spec, (tuple, list)) else (spec, 1.0)
    if strength <= 0:
        return v
    if mode == "noise":
        return v + strength * v.std() * torch.randn_like(v)
    repl = {"shuffle": lambda: v[torch.randperm(v.shape[0], device=v.device)],
            "mean": lambda: v.mean(0, keepdim=True).expand_as(v),
            "zero": lambda: torch.zeros_like(v)}[mode]()
    return (1.0 - strength) * v + strength * repl


class TwoRoute(nn.Module):
    CORES = {"gru": nn.GRU, "lstm": nn.LSTM, "rnn": nn.RNN}
    K = 10.0

    def __init__(self, cfg=None, core="gru", d_h=None, d_c=None, n_out=None,
                 attention=True, gate=False, exclude_last=True):
        super().__init__()
        cfg = cfg or CFG
        d_h = d_h or cfg.d_model
        d_c = d_c or d_h
        self.cfg, self.d_h, self.d_c = cfg, d_h, d_c
        self.attention, self.use_gate, self.exclude_last = attention, gate, exclude_last
        n_out = n_out or N_CLASSES
        self.emb = nn.Embedding(vocab_size(cfg), d_h)
        self.rnn = self.CORES[core](d_h, d_h, batch_first=True)
        self.proj_h = nn.Sequential(nn.Linear(d_h, d_h), nn.GELU())
        self.w_h = nn.Linear(d_h, n_out)
        if attention:
            self.q, self.k, self.v = (nn.Linear(d_h, d_c), nn.Linear(d_h, d_c),
                                      nn.Linear(d_h, d_c))
            self.scale = d_c ** -0.5
            self.proj_c = nn.Sequential(nn.Linear(d_c, d_c), nn.GELU())
            self.w_c = nn.Linear(d_c, n_out, bias=False)
        if gate:
            self.gate_logit = nn.Parameter(torch.tensor(2.0))

    @property
    def z(self):
        return torch.sigmoid(self.gate_logit) if self.use_gate else None

    def attend(self, hq, H, window=None):
        B, T_, _ = H.shape
        sc = (self.q(hq).unsqueeze(1) @ self.k(H).transpose(1, 2)).squeeze(1) * self.scale
        if window is not None:
            idx = torch.arange(T_, device=H.device)
            sc = sc.masked_fill((idx < T_ - window).unsqueeze(0), float("-inf"))
        w = sc.softmax(-1)
        return torch.einsum("bt,btd->bd", w, self.v(H)), w

    def forward(self, x, intervene=None, return_internals=False):
        iv = intervene or {}
        H, _ = self.rnn(self.emb(x))
        h_T = H[:, -1, :]
        h_r = ablate(h_T, iv.get("hidden"))
        lh = self.w_h(self.proj_h(h_r))
        if not self.attention:
            out = {"logits": lh, "parts": {"hidden": lh, "context": torch.zeros_like(lh)}}
            if return_internals:
                out.update({"H": H, "h_T": h_T, "ctx": None, "attn": None})
            return out
        H_ctx = H[:, :-1, :] if self.exclude_last else H
        ctx, aw = self.attend(ablate(h_T, iv.get("query")), H_ctx,
                              window=iv.get("attn_window"))
        ctx = ablate(ctx, iv.get("context"))
        lc = self.w_c(self.proj_c(ctx))
        if self.use_gate:
            nrm = lambda u: u / (u.norm(dim=-1, keepdim=True) + 1e-6)
            lh, lc = self.K * nrm(lh), self.K * self.z * nrm(lc)
        out = {"logits": lh + lc, "parts": {"hidden": lh, "context": lc}}
        if return_internals:
            out.update({"H": H, "h_T": h_T, "ctx": ctx, "attn": aw})
        return out


def route_params(m):
    rec = list(m.proj_h.parameters()) + list(m.w_h.parameters())
    att = ((list(m.q.parameters()) + list(m.k.parameters()) + list(m.v.parameters())
            + list(m.proj_c.parameters()) + list(m.w_c.parameters()))
           if m.attention else [])
    if getattr(m, "use_gate", False):
        att = att + [m.gate_logit]
    shared = list(m.emb.parameters()) + list(m.rnn.parameters())
    return rec, att, shared


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


CORE_OF = {"gru": "gru", "gru_attn": "gru"}
HAS_ATTN = {"gru": False, "gru_attn": True}


def match_capacity(cfg=None):
    """Same rule as the core suite: reference is the attention-free GRU at d_model."""
    cfg = cfg or CFG
    target = count_params(TwoRoute(cfg, "gru", d_h=cfg.d_model, attention=False))
    dims = {name: min(range(8, 400, 4),
                      key=lambda d: abs(count_params(TwoRoute(
                          cfg, CORE_OF[name], d_h=d, attention=HAS_ATTN[name])) - target))
            for name in CORE_OF}
    return dims, target


DIMS, _TARGET = match_capacity()


def build(name="gru_attn", cfg=None, d_h=None, d_c=None, gate=False, n_out=None):
    return TwoRoute(cfg or CFG, CORE_OF[name], d_h=d_h or DIMS[name], d_c=d_c,
                    n_out=n_out, attention=HAS_ATTN[name], gate=gate)


_P = {k: count_params(build(k)) for k in CORE_OF}
print(f"capacity matching: target {_TARGET:,}; dims {DIMS}; params {_P}")
if _P["gru_attn"] != 33028 or DIMS["gru_attn"] != 52:
    print("!! differs from paper Table 3 (gru_attn d=52, 33,028) — check the config")

In [ ]:
def train(task, load, lam=0.0, form="l2", seed=0, steps=2500, model=None,
          name="gru_attn", d_h=None, d_c=None, gate=False, schedule="cosine",
          p_drop=0.0, log_every=None, cfg=None, handicap=None, lam_after=None,
          switch=None, ckpt_key=None, ckpt_every=2000):
    """Verbatim from the core suite."""
    cfg = cfg or CFG
    set_seed(seed)
    m = (model or build(name, cfg, d_h, d_c, gate, task.n_out)).to(DEVICE)
    rec, att, shared = route_params(m)
    groups = [{"params": rec, "name": "recurrent"}, {"params": shared, "name": "shared"}]
    if att:
        groups.insert(1, {"params": att, "name": "attention"})
    opt = torch.optim.AdamW(groups, lr=cfg.lr, weight_decay=cfg.weight_decay)
    _grouped = sum(len(g["params"]) for g in groups)
    _total = len([p for p in m.parameters() if p.requires_grad])
    assert _grouped == _total, "parameter tensor(s) missing from optimiser groups"
    trace, start = [], 1

    if ckpt_key is not None:
        ck = ckpt_load(f"run_{ckpt_key}")
        if ck is not None and ck.get("steps") == steps:
            m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            trace, start = ck["trace"], ck["step"] + 1
            if start <= steps:
                print(f"    resumed run {ckpt_key} at step {ck['step']}/{steps}")

    for step in range(start, steps + 1):
        cos = 0.5 * (1 + math.cos(math.pi * step / steps)) if schedule == "cosine" else 1.0
        for g in opt.param_groups:
            f = (handicap["factor"] if handicap and step <= handicap["until"]
                 and g["name"] == handicap["route"] else 1.0)
            g["lr"] = cfg.lr * cos * f
        cur = lam if (switch is None or step <= switch) else lam_after
        m.train()
        b = task.generate(cfg.batch_size, load)
        iv = ({"context": ("zero", 1.0)} if p_drop > 0 and random.random() < p_drop
              else None)
        o = m(b["x"], intervene=iv)
        loss = F.cross_entropy(o["logits"], b["y"])
        if cur and cur > 0 and m.attention:
            lc = o["parts"]["context"]
            loss = loss + cur * ({"l2": lambda: lc.norm(dim=-1).mean(),
                                  "l2sq": lambda: lc.pow(2).sum(-1).mean(),
                                  "gate": lambda: m.z}[form]())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), cfg.grad_clip)
        opt.step()
        if log_every and (step % log_every == 0 or step == 1):
            trace.append({"step": step, **deps(m, task, load, cfg)})
        if ckpt_key is not None and step % ckpt_every == 0 and step < steps:
            ckpt_save(f"run_{ckpt_key}",
                      {"model": m.state_dict(), "opt": opt.state_dict(),
                       "step": step, "steps": steps, "trace": trace})

    if ckpt_key is not None:
        p = ckpt_path(f"run_{ckpt_key}")
        if p.exists():
            p.unlink()
    return m, pd.DataFrame(trace)


@torch.no_grad()
def accuracy(m, task, load, intervene=None, n_batches=8, cfg=None, **gen):
    cfg = cfg or CFG
    m.eval(); c = t = 0
    for _ in range(n_batches):
        b = task.generate(cfg.batch_size, load, **gen)
        c += (m(b["x"], intervene=intervene)["logits"].argmax(-1) == b["y"]).sum().item()
        t += b["y"].numel()
    m.train(); return c / t


@torch.no_grad()
def deps(m, task, load, cfg=None, n_batches=8, mode="shuffle"):
    """Verbatim: resampling dependence normalised by headroom above chance."""
    cfg = cfg or CFG
    chance = 1.0 / m.w_h.out_features
    acc = accuracy(m, task, load, n_batches=n_batches, cfg=cfg)
    den = max(acc - chance, 1e-6)
    d = lambda a: float(np.clip((acc - a) / den, 0, 1))
    hd = d(accuracy(m, task, load, {"hidden": (mode, 1.0)}, n_batches, cfg))
    if not m.attention:
        return {"acc_clean": acc, "hidden_dep": hd, "attn_dep": 0.0, "context_share": 0.0}
    ad = d(accuracy(m, task, load, {"context": (mode, 1.0)}, n_batches, cfg))
    m.eval(); sh, sc = [], []
    for _ in range(4):
        p = m(task.generate(cfg.batch_size, load)["x"])["parts"]
        sh.append(p["hidden"].std(-1)); sc.append(p["context"].std(-1))
    m.train()
    h, c = torch.cat(sh).mean().item(), torch.cat(sc).mean().item()
    return {"acc_clean": acc, "hidden_dep": hd, "attn_dep": ad,
            "context_share": c / (h + c + 1e-9)}


def crossing_any(x, hidden, attn):
    """Verbatim."""
    x = np.asarray(x, float)
    d = np.asarray(attn, float) - np.asarray(hidden, float)
    ok = ~np.isnan(d); x, d = x[ok], d[ok]
    o = np.argsort(x); x, d = x[o], d[o]
    for i in range(len(d) - 1):
        a, b = d[i], d[i + 1]
        if a == 0:
            return float(x[i])
        if a * b < 0:
            f = abs(a) / (abs(a) + abs(b))
            return float(x[i] + f * (x[i + 1] - x[i]))
    return np.nan


# ------------------------------ new measurements ------------------------------
@torch.no_grad()
def acc_parts(m, task, load, n_batches=8, cfg=None):
    """Clean accuracy and accuracy with each route's logits DELETED."""
    cfg = cfg or CFG
    m.eval(); c = {"clean": 0, "no_ret": 0, "no_rec": 0}; t = 0
    for _ in range(n_batches):
        b = task.generate(cfg.batch_size, load)
        p = m(b["x"])["parts"]; y = b["y"]
        c["clean"] += ((p["hidden"] + p["context"]).argmax(-1) == y).sum().item()
        c["no_ret"] += (p["hidden"].argmax(-1) == y).sum().item()
        c["no_rec"] += (p["context"].argmax(-1) == y).sum().item()
        t += y.numel()
    m.train()
    return {k: v / t for k, v in c.items()}


def route_eval(m, task, load, n_batches=8):
    """Resampling dependence (paper metric) + deletion necessity."""
    d = deps(m, task, load, n_batches=n_batches)
    if not m.attention:
        return d
    a = acc_parts(m, task, load, n_batches)
    head = max(a["clean"] - 1.0 / m.w_h.out_features, 1e-6)
    d.update(acc_no_ret=a["no_ret"], acc_no_rec=a["no_rec"],
             del_ret=float(np.clip((a["clean"] - a["no_ret"]) / head, 0, 1)),
             del_rec=float(np.clip((a["clean"] - a["no_rec"]) / head, 0, 1)))
    return d


@torch.no_grad()
def attn_stats(m, task, load, n_batches=8):
    """How the retrieval read is distributed over the K/V positions.

    eff_pos         exp(entropy) of the attention weights: 1 = one-position pointer
    eff_frac_items  eff_pos / load: ~1/load for a pointer, ~1 for pooling over items
    top1_mass       weight on the single most-attended position
    item_mass       total weight on item positions (vs delay positions)
    last_kv_mass    weight on the last K/V state h_{T-1}, which already holds the
                    recurrent summary; high values mean retrieval is re-reading the
                    recurrent state rather than reading the items
    key_mass        (selective recall) weight on the queried position
    mode_enrichment (majority) share of item-mass on modal-symbol positions divided
                    by the modal symbol's share of positions; 1 = unweighted pooling
    """
    if not m.attention:
        return {}
    m.eval(); acc = {}
    for _ in range(n_batches):
        b = task.generate(CFG.batch_size, load)
        w = m(b["x"], return_internals=True)["attn"]
        ent = -(w.clamp_min(1e-12).log() * w).sum(-1)
        eff = ent.exp()
        item = w[:, :load].sum(-1)
        r = {"eff_pos": eff, "eff_frac_items": eff / load, "top1_mass": w.max(-1).values,
             "item_mass": item, "last_kv_mass": w[:, -1]}
        if b.get("key_pos") is not None:
            r["key_mass"] = w.gather(1, b["key_pos"].unsqueeze(1)).squeeze(1)
        if "counts" in b:
            is_mode = (b["items"] == b["y"].unsqueeze(1)).float()
            mm = (w[:, :load] * is_mode).sum(-1) / item.clamp_min(1e-9)
            r["mode_enrichment"] = mm / is_mode.mean(-1)
        for k, v in r.items():
            acc.setdefault(k, []).append(v.float().cpu())
    m.train()
    return {k: float(torch.cat(v).mean()) for k, v in acc.items()}


def fmt(g, col, d=3):
    return f"{g[col].mean():.{d}f}±{g[col].std(ddof=1) if len(g) > 1 else 0:.{d}f}"

## Phase 1 — Screen (the go / no-go)

Three arms per load, same as the paper's §5.1 premise: the free hybrid at the default budget, and the attention-free GRU (capacity-matched) at the default budget and at 3× (recurrence is the slow solution to find). Deletion and attention structure are recorded for the free hybrid.

In [ ]:
if RUN["screen"]:
    st = Store("screen")
    ARMS = [("hybrid_free", "gru_attn", STEPS), ("recurrent_only", "gru", STEPS),
            ("recurrent_only_long", "gru", REC_LONG)]
    units = list(itertools.product(SCREEN_GRID, ARMS, SEEDS_SCREEN))
    t0 = time.time()
    try:
      for L, (arm, name, steps), sd in units:
        spec = {"block": "screen", "task": "majority", "arm": arm, "load": L, "seed": sd,
                "steps": steps, "margin": MARGIN, "cfg": cfg_tag()}

        def _go(L=L, name=name, steps=steps, sd=sd):
            task = make_task("majority")
            m, _ = train(task, L, lam=0.0, seed=sd, steps=steps, name=name)
            out = route_eval(m, task, L)
            out.update({f"attn_{k}": v for k, v in attn_stats(m, task, L).items()})
            return out

        cached(st, spec, _go, label=f"{arm:<20} L={L:<3} seed {sd}")
    except KeyboardInterrupt:
        print("  interrupted — finished runs are saved; loads with every seed done are analysed")
    SCR = st.frame(); SCR.to_csv(csv("screen"), index=False)
    print(f"\nscreen: {len(SCR)} runs on disk ({(time.time()-t0)/60:.1f} min this session)")

In [ ]:
SCR_ALL = pd.read_csv(csv("screen"))
SCR = SCR_ALL[SCR_ALL.load.isin(LOAD_GRID)]          # decision uses the pre-declared grid only
REC_COL = f"rec_only@{REC_LONG}"
_n = SCR.groupby(["load", "arm"]).seed.nunique().unstack()
_complete = _n.index[(_n.reindex(columns=["hybrid_free", "recurrent_only",
                                            "recurrent_only_long"]) >= len(SEEDS_SCREEN)).all(axis=1)]
SCR = SCR[SCR.load.isin(_complete)]


def _agg(arm, col, how="mean"):
    g = SCR[SCR.arm == arm].groupby("load")[col]
    return getattr(g, how)()


T = pd.DataFrame({
    f"rec_only@{STEPS}": _agg("recurrent_only", "acc_clean"),
    REC_COL:             _agg("recurrent_only_long", "acc_clean"),
    "hyb_acc":           _agg("hybrid_free", "acc_clean"),
    "hyb_Dh":            _agg("hybrid_free", "hidden_dep"),
    "hyb_Dc":            _agg("hybrid_free", "attn_dep"),
    "hyb_Dc_min":        _agg("hybrid_free", "attn_dep", "min"),
    "hyb_share":         _agg("hybrid_free", "context_share"),
    "hyb_acc_no_ret":    _agg("hybrid_free", "acc_no_ret"),
    "hyb_del_ret":       _agg("hybrid_free", "del_ret"),
    "eff_frac_items":    _agg("hybrid_free", "attn_eff_frac_items"),
    "item_mass":         _agg("hybrid_free", "attn_item_mass"),
    "last_kv_mass":      _agg("hybrid_free", "attn_last_kv_mass"),
})
C1 = (T.hyb_Dc >= DC_MIN) & (T.hyb_Dh <= DH_MAX)
C2 = T[REC_COL] >= VIABLE_ACC
C3 = T.hyb_del_ret >= DELETION_MIN
SOLVED = T.hyb_acc >= VIABLE_ACC
T["C1_ret_dom"], T["C2_rec_viable"], T["C3_ret_needed"] = C1, C2, C3
pd.set_option("display.width", 200)
print("SCREEN over the pre-declared grid (means over seeds)\n")
print(T.round(3).to_string())

pos = [L for L in T.index if C1[L] and C2[L] and C3[L] and SOLVED[L]]
L_pos = int(max(pos)) if pos else None
neg = [L for L in T.index if L_pos is not None and L > L_pos and not C2[L]
       and C1[L] and T.hyb_acc[L] >= ACC_FLOOR]
L_neg = int(min(neg)) if neg else None

reasons = []
if not C2.any():
    reasons.append("recurrence is not viable at any load: no substitute exists")
if C2.all():
    reasons.append("recurrence viable at every pre-declared load: no boundary control on "
                   "this grid (searched in Section 6)")
if not (C1 & SOLVED).any():
    reasons.append("free hybrid is never retrieval-dominant: nothing to hand over")
if (C1 & SOLVED).any() and not C3.any():
    reasons.append("retrieval is never load-bearing (deletion is cheap)")
GO = L_pos is not None
print(f"\nL_pos = {L_pos}   L_neg (pre-declared grid) = {L_neg}   DECISION: {'GO' if GO else 'NO-GO'}")
for r in reasons:
    print(f"  - {r}")
if GO:
    print(f"structural check at L_pos: item_mass {T.item_mass[L_pos]:.3f}, "
          f"last_kv_mass {T.last_kv_mass[L_pos]:.3f} "
          f"({'reads items' if T.last_kv_mass[L_pos] <= 0.5 else 'reads the recurrent summary'})")

sel = load_sel()
if "L_pos" not in sel:
    sel = {"task": "majority", "margin": MARGIN, "n_symbols": N_SYMBOLS,
           "load_grid": LOAD_GRID, "L_pos": L_pos, "L_neg": L_neg, "go": GO,
           "no_go_reasons": reasons,
           "thresholds": {"VIABLE_ACC": VIABLE_ACC, "ACC_FLOOR": ACC_FLOOR,
                          "DC_MIN": DC_MIN, "DH_MAX": DH_MAX,
                          "DELETION_MIN": DELETION_MIN, "LAM_MULT": LAM_MULT},
           "screen_budget": {"hybrid": STEPS, "recurrence_viability": REC_LONG},
           "screen_seeds": SEEDS_SCREEN, "preset": PRESET,
           "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    if PRESET == "smoke" and not GO:
        sel.update(L_pos=LOAD_GRID[1], note="SMOKE placeholder selection; not a result")
    save_sel(sel)
    print(f"\nselection frozen -> {SEL_PATH}")
else:
    print(f"\nselection already frozen at {sel.get('frozen_at')} "
          f"(L_pos={sel['L_pos']}); not re-decided")
T.round(4).to_csv(csv("screen_summary"))

## Phase 2 — Price selection at `L_pos`

Priced-from-initialisation dose–response (default 2500-step cosine budget, the same budget the boundary sweeps used). λ\* is the per-seed Dh = Dc crossing on the log-λ axis above the accuracy floor (geometric mean over seeds). The four-arm price is `LAM_MULT × λ*`, the pre-specified 3× rule from App. C, capped at the grid maximum.

In [ ]:
sel = load_sel(); GO = go_flag(sel)
if RUN["price"] and GO and sel.get("L_pos") is not None:
    L = sel["L_pos"]; task = make_task("majority"); st = Store("price")
    for lam, sd in itertools.product(LAM_GRID, SEEDS_SCREEN):
        spec = {"block": "price", "load": L, "lam": lam, "seed": sd, "steps": STEPS,
                "margin": MARGIN}

        def _go(lam=lam, sd=sd):
            m, _ = train(task, L, lam=lam, seed=sd, steps=STEPS)
            return route_eval(m, task, L)

        cached(st, spec, _go, label=f"lam={lam:<6} seed {sd}")
    PR = st.frame(); PR.to_csv(csv("price"), index=False)
    print(f"\ndose-response at L={L} (priced from init, {STEPS} steps):")
    print(PR.groupby("lam")[["acc_clean", "hidden_dep", "attn_dep", "context_share",
                             "del_ret"]].mean().round(3).to_string())

    cr = []
    for sd, gg in PR[PR.acc_clean >= ACC_FLOOR].groupby("seed"):
        gg = gg.sort_values("lam")
        c = crossing_any(np.log2(gg.lam.values), gg.hidden_dep.values, gg.attn_dep.values)
        cr.append(float(2.0 ** c) if c == c else np.nan)
    ok = [c for c in cr if c == c]
    print(f"\nper-seed lambda* : {np.round(cr, 4).tolist()}")
    if "lam" not in sel:
        if ok:
            lam_star = float(np.exp(np.mean(np.log(ok))))
            sel.update(lam_star=lam_star, lam=float(min(LAM_MULT * lam_star, max(LAM_GRID))),
                       lam_crossings=cr, lam_rule=f"{LAM_MULT}x geometric-mean lambda*, "
                       f"capped at {max(LAM_GRID)}")
        elif PRESET == "smoke":
            sel.update(lam=0.1, lam_star=None, lam_rule="SMOKE placeholder")
        else:
            sel.update(lam=None, lam_star=None, go=False,
                       lam_rule="no Dh=Dc crossing above the accuracy floor on the grid")
            print("  NO-GO: retrieval cannot be dislodged at acceptable accuracy at L_pos")
        save_sel(sel)
    print(f"selected: lambda* = {sel.get('lam_star')}, four-arm lambda = {sel.get('lam')}  "
          f"({sel.get('lam_rule')})")
else:
    print("price selection skipped (NO-GO or disabled)")

## Phase 3 — Switch timing from an unpriced pilot

Following the disclosed large-scale procedure (App. B), the switch step for each load is set from the **pre-intervention state only**. It is the first logged step, stable over two consecutive logs, at which the unpriced model is solved (acc ≥ `ACC_FLOOR`) **and** retrieval-dominant. The maximum is taken over endpoint seeds, plus one logging interval, and the result is never earlier than `SW_MIN` (paper: 2000). The pilot uses the same seed and data stream as the arms, so its pre-switch trajectory is the arms' pre-switch trajectory; the arms also record their own state at the switch as a direct check.

In [ ]:
sel = load_sel(); GO = go_flag(sel)
LOADS_4A = [L for L in (sel.get("L_pos"), sel.get("L_neg")) if L is not None]
READY = RUN["four_arm"] and GO and sel.get("lam") is not None and len(LOADS_4A) > 0

if READY:
    st = Store("pilot")
    for L, sd in itertools.product(LOADS_4A, SEEDS_END):
        spec = {"block": "pilot", "load": L, "seed": sd, "steps": PILOT, "log": LOG,
                "margin": MARGIN}

        def _go(L=L, sd=sd, spec=spec):
            task = make_task("majority")
            m, tr = train(task, L, lam=0.0, seed=sd, steps=PILOT, schedule="constant",
                          log_every=LOG, ckpt_key=keyof(spec))
            return {"trace": tr.to_dict("records")}

        cached(st, spec, _go, label=f"pilot L={L} seed {sd}")
    PT = st.traces()

    def converged_step(g):
        g = g.sort_values("step").reset_index(drop=True)
        ok = ((g.acc_clean >= ACC_FLOOR) & (g.attn_dep >= DC_MIN)
              & (g.hidden_dep <= DH_MAX)).values
        for i in range(len(ok) - 1):
            if ok[i] and ok[i + 1]:
                return int(g.step[i])
        return None

    SWD, NC = dict(sel.get("switch", {})), dict(sel.get("pilot_not_converged", {}))
    if any(str(L) not in SWD for L in LOADS_4A):
        for L in LOADS_4A:
            if str(L) in SWD:
                continue
            steps = {sd: converged_step(PT[(PT.load == L) & (PT.seed == sd)])
                     for sd in SEEDS_END}
            good = [s for s in steps.values() if s is not None]
            sw = max(SW_MIN, (max(good) if good else PILOT) + LOG)
            SWD[str(L)] = int(math.ceil(sw / LOG) * LOG)
            NC[str(L)] = [sd for sd, s in steps.items() if s is None]
            print(f"  L={L}: converged step per seed {steps} -> switch at {SWD[str(L)]}")
        sel.update(switch=SWD, pilot_not_converged=NC,
                   switch_rule="first stable solved+retrieval-dominant logged step of the "
                               "unpriced pilot, max over seeds, +1 log interval, >= SW_MIN")
        save_sel(sel)
    print(f"\nswitch steps: {sel['switch']}   seeds not converged in pilot: "
          f"{sel.get('pilot_not_converged')}")
else:
    print("pilot skipped (NO-GO, no price, or disabled)")

## Section 4 — Four-arm switch at `L_pos`

Constant learning rate, `POST` steps after the switch, and deletion at every endpoint. Final models are checkpointed for the controls. The boundary-control arms, if a control exists, run in Section 6 with identical specifications.

In [ ]:
ARM_PRICES = lambda lam: {"never": (0.0, 0.0), "late": (0.0, lam),
                          "maintained": (lam, lam), "withdrawn": (lam, 0.0)}
ARM_ORDER = ["never", "late", "maintained", "withdrawn"]

if READY:
    sel = load_sel(); lam = sel["lam"]; st = Store("four_arm"); t0 = time.time()
    units = [(L, arm, sd) for L in LOADS_4A
             for arm in (ARM_ORDER if L == sel["L_pos"] else NEG_ARMS) for sd in SEEDS_END]
    for L, arm, sd in units:
        SW = sel["switch"][str(L)]; ST = SW + POST
        lb, la = ARM_PRICES(lam)[arm]
        spec = {"block": "four_arm", "arm": arm, "load": L, "seed": sd, "lam": lam,
                "switch": SW, "steps": ST, "log": LOG, "margin": MARGIN}

        def _go(L=L, arm=arm, sd=sd, SW=SW, ST=ST, lb=lb, la=la, spec=spec):
            task = make_task("majority")
            m, tr = train(task, L, lam=lb, lam_after=la, switch=SW, seed=sd, steps=ST,
                          schedule="constant", log_every=LOG, ckpt_key=keyof(spec))
            save_model(f"fourarm_{arm}_L{L}_s{sd}", m)
            pre, post = tr[tr.step <= SW].iloc[-1], tr[tr.step > SW]
            out = {"acc_at_switch": float(pre.acc_clean), "Dh_at_switch": float(pre.hidden_dep),
                   "Dc_at_switch": float(pre.attn_dep),
                   "share_at_switch": float(pre.context_share),
                   "min_acc_post": float(post.acc_clean.min()) if len(post) else np.nan,
                   **route_eval(m, task, L)}
            out.update({f"attn_{k}": v for k, v in attn_stats(m, task, L).items()})
            out["trace"] = tr.to_dict("records")
            return out

        cached(st, spec, _go, label=f"L={L:<3} {arm:<10} seed {sd}")
    print(f"four-arm done ({(time.time()-t0)/60:.1f} min this session)")

In [ ]:
FA = Store("four_arm").frame()
if len(FA):
    sel = load_sel()
    FA = FA[FA.lam == sel.get("lam")]
    FA["inverted"] = FA.hidden_dep > FA.attn_dep
    FA["complete"] = (FA.hidden_dep >= 0.9) & (FA.attn_dep <= 0.1)
    FA["del_safe"] = (FA.acc_clean - FA.acc_no_ret) <= DEL_SAFE
    FA.to_csv(csv("four_arm"), index=False)
    rows = []
    for L in LOADS_4A:
        for arm in ARM_ORDER:
            g = FA[(FA.load == L) & (FA.arm == arm)]
            if not len(g):
                continue
            rows.append({"load": L, "arm": arm, "switch": sel["switch"][str(L)],
                         "acc": fmt(g, "acc_clean"), "Dh": fmt(g, "hidden_dep", 2),
                         "Dc": fmt(g, "attn_dep", 2), "share": fmt(g, "context_share", 3),
                         "inversion": f"{int(g.inverted.sum())}/{len(g)}",
                         "complete": f"{int(g.complete.sum())}/{len(g)}",
                         "acc_ret_deleted": fmt(g, "acc_no_ret"),
                         "deletable": f"{int(g.del_safe.sum())}/{len(g)}",
                         "min_acc_post": f"{g.min_acc_post.min():.3f}"})
    TAB = pd.DataFrame(rows)
    print(f"FOUR-ARM ENDPOINTS  (lambda = {sel['lam']:.4g}, {POST} steps after switch)\n")
    print(TAB.to_string(index=False))
    TAB.to_csv(csv("four_arm_table"), index=False)

    # was every arm actually retrieval-dominant and solved when the price changed?
    pre = FA[FA.arm.isin(["never", "late"])]
    bad = pre[(pre.acc_at_switch < ACC_FLOOR) | (pre.Dc_at_switch < DC_MIN)]
    print("\npre-switch state (never/late arms): acc "
          f"{pre.acc_at_switch.min():.3f}-{pre.acc_at_switch.max():.3f}, Dc "
          f"{pre.Dc_at_switch.min():.2f}-{pre.Dc_at_switch.max():.2f}")
    if len(bad):
        print("  !! not solved/retrieval-dominant at the switch (disclose these seeds):")
        print(bad[["load", "arm", "seed", "acc_at_switch", "Dc_at_switch"]].to_string(index=False))
    else:
        print("  every never/late run was solved and retrieval-dominant at the switch")
else:
    print("no four-arm results yet")

## Section 5a — Matched post-hoc attenuation to the steered endpoints (§4.2)

The free model is the *never* endpoint. Its retrieval logits are scaled at inference until its retrieval share matches the *maintained* and *late* endpoints. On Majority those endpoints reach share 0.000, so the matched scale is α ≈ 0 and this control **reduces to deletion**. It is kept for completeness; Section 5b gives the informative version.

In [ ]:
sel = load_sel()
if RUN["posthoc"] and READY and sel.get("L_pos") is not None:
    L = sel["L_pos"]; task = make_task("majority"); st = Store("posthoc")

    def share_at(m, alpha, n=8):
        m.eval(); sh, sc = [], []
        with torch.no_grad():
            for _ in range(n):
                o = m(task.generate(CFG.batch_size, L)["x"])
                sh.append(o["parts"]["hidden"].std(-1))
                sc.append(alpha * o["parts"]["context"].std(-1))
        m.train()
        h, c = torch.cat(sh).mean().item(), torch.cat(sc).mean().item()
        return c / (h + c + 1e-9)

    @torch.no_grad()
    def scaled_acc(m, alpha, intervene=None, n=8):
        m.eval(); c = t = 0
        for _ in range(n):
            b = task.generate(CFG.batch_size, L)
            o = m(b["x"], intervene=intervene)
            lg = o["parts"]["hidden"] + alpha * o["parts"]["context"]
            c += (lg.argmax(-1) == b["y"]).sum().item(); t += b["y"].numel()
        m.train(); return c / t

    for sd in SEEDS_END:
        spec = {"block": "posthoc", "load": L, "seed": sd, "lam": sel["lam"],
                "switch": sel["switch"][str(L)], "post": POST}

        def _go(sd=sd):
            ms = {}
            for arm in ["never", "maintained", "late"]:
                m = build("gru_attn", n_out=task.n_out).to(DEVICE)
                if not load_model_into(f"fourarm_{arm}_L{L}_s{sd}", m):
                    raise FileNotFoundError(f"fourarm_{arm}_L{L}_s{sd}")
                ms[arm] = m
            free = ms["never"]; d_f = route_eval(free, task, L)
            out = {"free_share": d_f["context_share"], "free_acc": d_f["acc_clean"],
                   "free_Dh": d_f["hidden_dep"]}
            chance = 1.0 / free.w_h.out_features
            for ref in ["maintained", "late"]:
                d_s = route_eval(ms[ref], task, L)
                lo, hi = 0.0, 1.0
                for _ in range(30):
                    mid = (lo + hi) / 2
                    if share_at(free, mid) > d_s["context_share"]:
                        hi = mid
                    else:
                        lo = mid
                alpha = (lo + hi) / 2
                acc = scaled_acc(free, alpha)
                if acc - chance > 0.2:
                    den = acc - chance
                    dh = float(np.clip((acc - scaled_acc(free, alpha,
                                        {"hidden": ("shuffle", 1.0)})) / den, 0, 1))
                else:
                    dh = np.nan
                out.update({f"{ref}_share": d_s["context_share"], f"{ref}_acc": d_s["acc_clean"],
                            f"{ref}_Dh": d_s["hidden_dep"], f"scaled_{ref}_alpha": alpha,
                            f"scaled_{ref}_share": share_at(free, alpha),
                            f"scaled_{ref}_acc": acc, f"scaled_{ref}_Dh": dh})
            return out

        try:
            cached(st, spec, _go, label=f"seed {sd}")
        except FileNotFoundError as e:
            print(f"  seed {sd}: missing checkpoint {e} — rerun the four-arm cell with RESUME "
                  f"after deleting that run's record, or skip this seed")
    PH = st.frame()
    if len(PH):
        PH.to_csv(csv("posthoc"), index=False)
        print(f"\nMATCHED POST-HOC ATTENUATION at L={L} ({len(PH)} seeds)\n")
        print(f"  {'condition':<34}{'share':>14}{'accuracy':>16}{'Dh':>14}")
        print(f"  {'free (never arm), unmodified':<34}{fmt(PH,'free_share'):>14}"
              f"{fmt(PH,'free_acc',4):>16}{fmt(PH,'free_Dh'):>14}")
        for ref in ["maintained", "late"]:
            print(f"  {'free, scaled to ' + ref:<34}{fmt(PH,f'scaled_{ref}_share'):>14}"
                  f"{fmt(PH,f'scaled_{ref}_acc',4):>16}"
                  f"{(fmt(PH,f'scaled_{ref}_Dh') if PH[f'scaled_{ref}_Dh'].notna().all() else 'n/i'):>14}")
            print(f"  {'steered: ' + ref:<34}{fmt(PH,f'{ref}_share'):>14}"
                  f"{fmt(PH,f'{ref}_acc',4):>16}{fmt(PH,f'{ref}_Dh'):>14}")
        print(f"\n  alpha (maintained match) = {PH.scaled_maintained_alpha.mean():.4f} "
              f"± {PH.scaled_maintained_alpha.std():.4f}   (n/i = not interpreted, near chance)")

## Section 5b — Matched post-hoc attenuation at non-zero shares (Table R3, Fig. R1g)

The free model is scaled to the retrieval shares reached by priced-from-initialisation models in the Section 2 dose–response. Those models play the role of the paper's maintained-price reference.

In [ ]:
# Matched post-hoc attenuation at NON-ZERO shares (Table R3, Fig. R1g).
# The steered endpoints reach share 0.000, where scaling equals deletion (cell above).
# Here the free (never-arm) model is scaled to the shares of priced-from-init models
# from the Section 2 dose-response, the analogue of the paper's maintained reference.
sel = load_sel()
if RUN["posthoc_curve"] and sel.get("lam") is not None and os.path.exists(csv("price")):
    L = sel["L_pos"]; task = make_task("majority"); st = Store("posthoc_curve")
    PR = pd.read_csv(csv("price"))
    REF = PR[PR.lam.isin(POSTHOC_REF_LAMS)].groupby("lam")[["context_share", "acc_clean"]].mean()
    for sd in SEEDS_END:
        spec = {"block": "posthoc_curve", "load": L, "seed": sd, "lam": sel["lam"],
                "ref_lams": POSTHOC_REF_LAMS, "n_batches": 32}

        def _go(sd=sd):
            m = build("gru_attn", n_out=task.n_out).to(DEVICE)
            if not load_model_into(f"fourarm_never_L{L}_s{sd}", m):
                raise FileNotFoundError(f"fourarm_never_L{L}_s{sd}")
            set_seed(100 + sd); m.eval(); LH, LC, Y = [], [], []
            with torch.no_grad():
                for _ in range(32):
                    b = task.generate(CFG.batch_size, L); p = m(b["x"])["parts"]
                    LH.append(p["hidden"]); LC.append(p["context"]); Y.append(b["y"])
            lh, lc, y = torch.cat(LH), torch.cat(LC), torch.cat(Y)
            sh, sc = lh.std(-1).mean().item(), lc.std(-1).mean().item()
            rows = []
            for lam_ref, r in REF.iterrows():
                t = float(r.context_share); a = t * sh / ((1 - t) * sc)   # alpha giving share t
                rows.append({"lam_ref": float(lam_ref), "share": t, "alpha": a,
                             "scaled_acc": ((lh + a * lc).argmax(-1) == y).float().mean().item(),
                             "trained_acc": float(r.acc_clean)})
            return {"rows": rows}

        try:
            cached(st, spec, _go, label=f"seed {sd}")
        except FileNotFoundError as e:
            print(f"  seed {sd}: checkpoint {e} not found — needs the four-arm models")
    CURVE = st.frame()
    if len(CURVE):
        CURVE.to_csv(csv("posthoc_curve"), index=False)
        print("free model scaled to the share of a trained priced model vs the trained model\n")
        print(CURVE.groupby("lam_ref")[["share", "scaled_acc", "trained_acc"]]
              .agg(["mean", "std"]).round(4).to_string())

## Structural evidence — is the retrieval computation different from selective recall?

Same statistics on a free hybrid trained on selective recall at `min(L_pos, 24)`. The claim this cell supports: in selective recall the retrieval read is a near one-hot pointer to the queried position; in Majority it is distributed pooling over items. `mode_enrichment > 1` would mean the read is selectively weighted toward modal positions (the query h_T carries a candidate), which is still aggregation, not lookup.

In [ ]:
sel = load_sel()
if RUN["structure"] and sel.get("L_pos") is not None:
    L = sel["L_pos"]; L_ref = min(L, MAX_POS); st = Store("structure_ref")
    for sd in SEEDS_SCREEN:
        spec = {"block": "structure_ref", "task": "selective_recall", "load": L_ref,
                "seed": sd, "steps": STEPS}

        def _go(sd=sd):
            t = make_task("selective_recall")
            m, _ = train(t, L_ref, lam=0.0, seed=sd, steps=STEPS)
            return {**route_eval(m, t, L_ref),
                    **{f"attn_{k}": v for k, v in attn_stats(m, t, L_ref).items()}}

        cached(st, spec, _go, label=f"selective recall L={L_ref} seed {sd}")
    REF = st.frame()
    SCR = pd.read_csv(csv("screen"))
    MJ = SCR[(SCR.arm == "hybrid_free") & (SCR.load == L)]
    FA = pd.read_csv(csv("four_arm")) if os.path.exists(csv("four_arm")) else pd.DataFrame()
    NV = FA[(FA.load == L) & (FA.arm == "never")] if len(FA) else pd.DataFrame()

    cols = ["acc_clean", "attn_dep", "attn_eff_pos", "attn_eff_frac_items", "attn_top1_mass",
            "attn_item_mass", "attn_last_kv_mass", "attn_key_mass", "attn_mode_enrichment"]
    rows = []
    for lbl, g, LL in [(f"selective recall, free (L={L_ref})", REF, L_ref),
                       (f"majority, free screen (L={L})", MJ, L),
                       (f"majority, never arm endpoint (L={L})", NV, L)]:
        if len(g):
            rows.append({"model": lbl, "items": LL,
                         **{c: round(float(g[c].mean()), 3) for c in cols if c in g}})
    STRUCT = pd.DataFrame(rows).set_index("model")
    print("ATTENTION STRUCTURE (means over seeds)\n")
    print(STRUCT.to_string())
    STRUCT.to_csv(csv("structure"))
    if len(REF) and len(MJ):
        e_sr, e_mj = REF.attn_eff_pos.mean(), MJ.attn_eff_pos.mean()
        print(f"\n  effective attended positions: selective recall {e_sr:.2f} vs majority "
              f"{e_mj:.2f} ({e_mj/max(e_sr,1e-9):.1f}x)")
        print(f"  top-1 mass: {REF.attn_top1_mass.mean():.3f} vs {MJ.attn_top1_mass.mean():.3f}")

        fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.2))
        lab = ["sel. recall", "majority"]
        for a, c, t in zip(ax, ["attn_eff_frac_items", "attn_top1_mass", "attn_last_kv_mass"],
                           ["effective positions / items", "top-1 attention mass",
                            "mass on h_{T-1}"]):
            a.bar(lab, [REF[c].mean(), MJ[c].mean()],
                  yerr=[REF[c].std(), MJ[c].std()], color=["#8d99ae", "#1b6ca8"])
            a.set_title(t, fontsize=8); a.set_ylim(0, 1.05)
            a.spines[["top", "right"]].set_visible(False)
        fig.tight_layout()
        for ext in ("pdf", "png"):
            fig.savefig(FIGDIR / f"majority_attention_structure.{ext}", dpi=300,
                        bbox_inches="tight")
        plt.show()

## Section 5c — Length shift (Table R5)

Models trained at `L_pos` are evaluated at 1×, 1.5× and 2× length. The runner-up share is **pooled over errors**, because averaging per-seed rates would count seeds with zero errors as 0. Separation uses the paper's Table 18 rule. Error rates are not matched across configurations, so this is supporting evidence only. Stored results are loaded when present, so the reported numbers are reused exactly.

In [ ]:
# Length shift (Table R5): error rate of models trained at L_pos on longer sequences,
# and the fraction of errors that are a runner-up symbol. Loaded from disk when present.
sel = load_sel()
if RUN["length_shift"] and sel.get("L_pos") is not None:
    L = sel["L_pos"]; task = make_task("majority")

    @torch.no_grad()
    def runner_up_errors(m, load, n_batches=40):
        m.eval(); n_err = 0; hit = 0.0; ch = 0.0
        for _ in range(n_batches):
            b = task.generate(CFG.batch_size, load)
            pred = m(b["x"])["logits"].argmax(-1)
            wrong = pred != b["y"]
            if wrong.any():
                ru = b["runner_up"][wrong].float()
                hit += ru.gather(1, pred[wrong].unsqueeze(1)).sum().item()
                ch += (ru.sum(1) / (CFG.n_symbols - 1)).sum().item()
                n_err += int(wrong.sum())
        m.train()
        n = CFG.batch_size * n_batches
        return {"err_rate": n_err / n, "n_err": n_err,
                "runner_up_rate": hit / max(n_err, 1), "chance": ch / max(n_err, 1)}

    if RESUME and os.path.exists(csv("fingerprint")):
        FP = pd.read_csv(csv("fingerprint")); print(f"loaded {csv('fingerprint')}")
    else:
        rows = []
        for sd in SEEDS_END:
            for arm, cfgn in [("never", "retrieval"), ("late", "recurrence"),
                              ("maintained", "recurrence")]:
                m = build("gru_attn", n_out=task.n_out).to(DEVICE)
                if not load_model_into(f"fourarm_{arm}_L{L}_s{sd}", m):
                    continue
                for LL in sorted({L, int(round(1.5 * L)), 2 * L}):
                    set_seed(10_000 + 100 * sd + LL)
                    rows.append({"seed": sd, "arm": arm, "config": cfgn, "eval_load": LL,
                                 **runner_up_errors(m, LL)})
        FP = pd.DataFrame(rows)
        if len(FP):
            FP["excess"] = FP.runner_up_rate - FP.chance
            FP.to_csv(csv("fingerprint"), index=False)
    if len(FP):
        # pool over errors (a per-seed mean would average in seeds with zero errors)
        FP["_hits"], FP["_ch"] = FP.runner_up_rate * FP.n_err, FP.chance * FP.n_err
        agg = FP.groupby(["eval_load", "arm"]).agg(
            err_mean=("err_rate", "mean"), err_sd=("err_rate", "std"),
            n_err=("n_err", "sum"), hits=("_hits", "sum"), ch=("_ch", "sum"))
        agg["runner_up_share"] = agg.hits / agg.n_err.clip(lower=1)
        agg["chance"] = agg.ch / agg.n_err.clip(lower=1)
        print(agg[["err_mean", "err_sd", "n_err", "runner_up_share", "chance"]].round(4).to_string())
        print("\nseparation (Table 18 rule: |gap| > summed seed SDs), never vs recurrence-configured:")
        for LL in sorted(FP.eval_load.unique()):
            nv = FP[(FP.eval_load == LL) & (FP.arm == "never")].err_rate
            for a in ["late", "maintained"]:
                x = FP[(FP.eval_load == LL) & (FP.arm == a)].err_rate
                gap, sds = x.mean() - nv.mean(), x.std() + nv.std()
                print(f"  L={LL:<3} {a:<10} gap {gap:+.4f}  summed SD {sds:.4f}  "
                      f"separated: {abs(gap) > sds}")

## Optional — no-penalty availability intervention (§4.3)

Retrieval is made stochastically unavailable on a fraction of training steps, with no contribution term in the objective. This closes the Goodhart objection on the new task too. Enable it with `RUN["availability"] = True`.

In [ ]:
sel = load_sel()
if RUN["availability"] and sel.get("L_pos") is not None:
    L = sel["L_pos"]; task = make_task("majority"); st = Store("availability")
    for p, sd in itertools.product([0.0, 0.3, 0.5, 0.7], SEEDS_SCREEN):
        spec = {"block": "availability", "load": L, "p_drop": p, "seed": sd, "steps": STEPS}

        def _go(p=p, sd=sd):
            m, _ = train(task, L, lam=0.0, seed=sd, steps=STEPS, p_drop=p)
            return route_eval(m, task, L)

        cached(st, spec, _go, label=f"p_drop={p} seed {sd}")
    AV = st.frame(); AV.to_csv(csv("availability"), index=False)
    print(AV.groupby("p_drop")[["acc_clean", "context_share", "hidden_dep", "attn_dep",
                                "del_ret"]].mean().round(3).to_string())

## Section 6a — Boundary search on the length axis (Table R1, rows marked †)

The pre-declared grid had no non-viable load. This section screens `EXTRA_LOADS` one at a time with the same arms, budgets and specifications, and stops at the first load where recurrence alone is below 0.95 while the free hybrid is solved and retrieval-dominant. If such a load exists, it runs the pilot and `NEG_ARMS` there at the same price. In the reported run, no load up to 128 qualified.

In [ ]:
def boundary_length_axis():
    sel = load_sel()
    assert sel.get("L_pos") is not None and sel.get("lam") is not None and "switch" in sel, \
        "selection incomplete — is PAPER_WORKDIR pointing at the folder with results_majority/?"
    Lp, lam = sel["L_pos"], sel["lam"]
    print(f"frozen: L_pos={Lp}, lambda={lam:.4g}, switch={sel['switch']}, L_neg={sel.get('L_neg')}")

    SCREEN_ARMS = [("recurrent_only_long", "gru", REC_LONG), ("hybrid_free", "gru_attn", STEPS),
                   ("recurrent_only", "gru", STEPS)]


    def _load_summary(SCR, L):
        g = SCR[SCR.load == L]
        if any(g[g.arm == a].seed.nunique() < len(SEEDS_SCREEN) for a, _, _ in SCREEN_ARMS):
            return None
        h = g[g.arm == "hybrid_free"]
        return {"load": L, "rec_only_long": g[g.arm == "recurrent_only_long"].acc_clean.mean(),
                "rec_only": g[g.arm == "recurrent_only"].acc_clean.mean(),
                "hyb_acc": h.acc_clean.mean(), "hyb_Dh": h.hidden_dep.mean(),
                "hyb_Dc": h.attn_dep.mean(), "hyb_del_ret": h.del_ret.mean(),
                "eff_frac_items": h.attn_eff_frac_items.mean(),
                "last_kv_mass": h.attn_last_kv_mass.mean()}


    def _is_control(r):
        return (r["rec_only_long"] < VIABLE_ACC and r["hyb_Dc"] >= DC_MIN
                and r["hyb_Dh"] <= DH_MAX and r["hyb_acc"] >= ACC_FLOOR)


    # ---- 1. screen, load by load, stopping at the first qualifying control -------
    st = Store("screen"); ext_rows = []
    try:
        for L in EXTRA_LOADS:
            if sel.get("L_neg") is not None:
                break
            for (arm, name, steps), sd in itertools.product(SCREEN_ARMS, SEEDS_SCREEN):
                spec = {"block": "screen", "task": "majority", "arm": arm, "load": L, "seed": sd,
                        "steps": steps, "margin": MARGIN, "cfg": cfg_tag()}

                def _go(L=L, name=name, steps=steps, sd=sd):
                    task = make_task("majority")
                    m, _ = train(task, L, lam=0.0, seed=sd, steps=steps, name=name)
                    out = route_eval(m, task, L)
                    out.update({f"attn_{k}": v for k, v in attn_stats(m, task, L).items()})
                    return out

                cached(st, spec, _go, label=f"{arm:<20} L={L:<3} seed {sd}")
            r = _load_summary(st.frame(), L); ext_rows.append(r)
            print(f"  L={L}: recurrence alone {r['rec_only_long']:.3f} | hybrid acc {r['hyb_acc']:.3f}, "
                  f"Dh {r['hyb_Dh']:.2f}, Dc {r['hyb_Dc']:.2f}, deletion {r['hyb_del_ret']:.2f}")
            if _is_control(r):
                sel.update(L_neg=int(L), L_neg_added_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                           L_neg_rule=f"first load above the frozen L_pos={Lp} where recurrence alone "
                                      f"< {VIABLE_ACC} ({REC_LONG} steps) and the free hybrid is "
                                      f"solved (>= {ACC_FLOOR}) and retrieval-dominant",
                           extended_grid=[int(x["load"]) for x in ext_rows])
                save_sel(sel)
                print(f"\n  boundary control found: L_neg = {L}  (selection updated)")
            elif r["rec_only_long"] < VIABLE_ACC:
                print(f"  L={L}: recurrence non-viable but the free hybrid is not a clean "
                      f"retrieval-dominant solution here; continuing")
    except KeyboardInterrupt:
        print("  interrupted — finished runs are saved; rerun this cell to continue")
    SCR = st.frame(); SCR.to_csv(csv("screen"), index=False)

    Ln = sel.get("L_neg")
    if Ln is None:
        print("\nNo boundary control up to L =", EXTRA_LOADS[-1], "— report Majority as positive-only "
              "(recurrence viable over the whole tested range) and do not add a Table 2 row.")
    else:
        # ---- 2. switch timing at L_neg from the unpriced pilot --------------------
        def _converged_step(g):
            g = g.sort_values("step").reset_index(drop=True)
            ok = ((g.acc_clean >= ACC_FLOOR) & (g.attn_dep >= DC_MIN)
                  & (g.hidden_dep <= DH_MAX)).values
            for i in range(len(ok) - 1):
                if ok[i] and ok[i + 1]:
                    return int(g.step[i])
            return None

        st = Store("pilot")
        for sd in SEEDS_END:
            spec = {"block": "pilot", "load": Ln, "seed": sd, "steps": PILOT, "log": LOG,
                    "margin": MARGIN}

            def _go(sd=sd, spec=spec):
                task = make_task("majority")
                m, tr = train(task, Ln, lam=0.0, seed=sd, steps=PILOT, schedule="constant",
                              log_every=LOG, ckpt_key=keyof(spec))
                return {"trace": tr.to_dict("records")}

            cached(st, spec, _go, label=f"pilot L={Ln} seed {sd}")
        if str(Ln) not in sel["switch"]:
            PT = st.traces()
            steps_ = {sd: _converged_step(PT[(PT.load == Ln) & (PT.seed == sd)]) for sd in SEEDS_END}
            good = [s for s in steps_.values() if s is not None]
            sw = max(SW_MIN, (max(good) if good else PILOT) + LOG)
            sel["switch"][str(Ln)] = int(math.ceil(sw / LOG) * LOG)
            sel.setdefault("pilot_not_converged", {})[str(Ln)] = \
                [sd for sd, s in steps_.items() if s is None]
            save_sel(sel)
            print(f"  L={Ln}: converged step per seed {steps_} -> switch at {sel['switch'][str(Ln)]}")

        # ---- 3. four-arm at L_neg, same price, same spec as the Phase 2 cell ------
        ARM_PRICES = lambda lam: {"never": (0.0, 0.0), "late": (0.0, lam),
                                  "maintained": (lam, lam), "withdrawn": (lam, 0.0)}
        st = Store("four_arm")
        SW = sel["switch"][str(Ln)]; ST = SW + POST
        for arm, sd in itertools.product(NEG_ARMS, SEEDS_END):
            lb, la = ARM_PRICES(lam)[arm]
            spec = {"block": "four_arm", "arm": arm, "load": Ln, "seed": sd, "lam": lam,
                    "switch": SW, "steps": ST, "log": LOG, "margin": MARGIN}

            def _go(arm=arm, sd=sd, lb=lb, la=la, spec=spec):
                task = make_task("majority")
                m, tr = train(task, Ln, lam=lb, lam_after=la, switch=SW, seed=sd, steps=ST,
                              schedule="constant", log_every=LOG, ckpt_key=keyof(spec))
                save_model(f"fourarm_{arm}_L{Ln}_s{sd}", m)
                pre, post = tr[tr.step <= SW].iloc[-1], tr[tr.step > SW]
                out = {"acc_at_switch": float(pre.acc_clean), "Dh_at_switch": float(pre.hidden_dep),
                       "Dc_at_switch": float(pre.attn_dep),
                       "share_at_switch": float(pre.context_share),
                       "min_acc_post": float(post.acc_clean.min()) if len(post) else np.nan,
                       **route_eval(m, task, Ln)}
                out.update({f"attn_{k}": v for k, v in attn_stats(m, task, Ln).items()})
                out["trace"] = tr.to_dict("records")
                return out

            cached(st, spec, _go, label=f"L={Ln:<3} {arm:<10} seed {sd}")

        # ---- 4. immediate readout at L_neg ------------------------------------------
        FN = st.frame(); FN = FN[(FN.load == Ln) & (FN.lam == lam)]
        print(f"\nBOUNDARY CONTROL at L={Ln} (lambda {lam:.4g}, switch {SW})")
        for arm in NEG_ARMS:
            g = FN[FN.arm == arm]
            if len(g):
                print(f"  {arm:<10} acc {fmt(g,'acc_clean')}  Dh {fmt(g,'hidden_dep',2)}  "
                      f"Dc {fmt(g,'attn_dep',2)}  inversion {int((g.hidden_dep > g.attn_dep).sum())}"
                      f"/{len(g)}  retrieval deleted {fmt(g,'acc_no_ret')}  "
                      f"acc@switch min {g.acc_at_switch.min():.3f}")
        _fa = st.frame(); _fa[_fa.lam == lam].to_csv(csv("four_arm"), index=False)


if RUN["boundary_length"] and load_sel().get("lam") is not None:
    boundary_length_axis()

## Section 6b — Boundary search on the alphabet axis (Table R6)

This axis was declared after 6a found no control. It keeps `L = L_pos` and screens alphabet size `K`, which sets the number of counters recurrence must hold. The model width is unchanged; only the embedding and output layers grow with `K`. In the reported run, pooled retrieval degraded before recurrence, and no `K` qualified.

In [ ]:
def boundary_alphabet_axis():
    sel = load_sel()
    if sel.get("L_neg") is not None:
        print(f"length axis already gave a control (L_neg={sel['L_neg']}); alphabet axis not run")
        return
    Lp, lam = sel["L_pos"], sel["lam"]
    print(f"frozen positive: L={Lp}, K={N_SYMBOLS}, lambda={lam:.4g}; boundary so far: {sel.get('boundary_k')}")

    SCREEN_ARMS = [("recurrent_only_long", "gru", REC_LONG), ("hybrid_free", "gru_attn", STEPS),
                   ("recurrent_only", "gru", STEPS)]


    def _cfg(K):
        return Config(n_symbols=K)


    st = Store("screen_k"); rows_k = []
    try:
        for K in K_GRID:
            if sel.get("boundary_k"):
                break
            for (arm, name, steps), sd in itertools.product(SCREEN_ARMS, SEEDS_SCREEN):
                spec = {"block": "screen_k", "task": "majority", "arm": arm, "load": Lp,
                        "n_symbols": K, "seed": sd, "steps": steps, "margin": MARGIN}

                def _go(K=K, name=name, steps=steps, sd=sd):
                    c = _cfg(K); task = make_task("majority", cfg=c)
                    m, _ = train(task, Lp, lam=0.0, seed=sd, steps=steps, name=name, cfg=c)
                    out = route_eval(m, task, Lp)
                    out.update({f"attn_{k}": v for k, v in attn_stats(m, task, Lp).items()})
                    return out

                cached(st, spec, _go, label=f"{arm:<20} K={K:<3} seed {sd}")
            F_ = st.frame(); g = F_[F_.n_symbols == K]; h = g[g.arm == "hybrid_free"]
            r = {"K": K, "chance": 1 / K,
                 "rec_only_long": g[g.arm == "recurrent_only_long"].acc_clean.mean(),
                 "hyb_acc": h.acc_clean.mean(), "hyb_Dh": h.hidden_dep.mean(),
                 "hyb_Dc": h.attn_dep.mean(), "hyb_del_ret": h.del_ret.mean(),
                 "eff_frac_items": h.attn_eff_frac_items.mean()}
            rows_k.append(r)
            print(f"  K={K}: recurrence alone {r['rec_only_long']:.3f} | hybrid acc {r['hyb_acc']:.3f}, "
                  f"Dh {r['hyb_Dh']:.2f}, Dc {r['hyb_Dc']:.2f}, deletion {r['hyb_del_ret']:.2f} "
                  f"(chance {1/K:.3f})")
            if (r["rec_only_long"] < VIABLE_ACC and r["hyb_Dc"] >= DC_MIN and r["hyb_Dh"] <= DH_MAX
                    and r["hyb_acc"] >= ACC_FLOOR):
                sel["boundary_k"] = {"axis": "alphabet size K", "L": Lp, "K_neg": K,
                                     "added_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                                     "rule": f"length axis gave no control up to L=128; first K in "
                                             f"{K_GRID} at L={Lp} with recurrence alone < {VIABLE_ACC} "
                                             f"and free hybrid solved (>= {ACC_FLOOR}) and retrieval-dominant",
                                     "screen": rows_k}
                save_sel(sel); print(f"\n  boundary control found: K = {K} at L = {Lp}")
    except KeyboardInterrupt:
        print("  interrupted — finished runs are saved; rerun this cell to continue")
    st.frame().to_csv(csv("screen_k"), index=False)

    bk = sel.get("boundary_k")
    if not bk:
        print(f"\nNo control up to K={K_GRID[-1]} — Majority is positive-only (no Table 2 row).")
    else:
        Kn = bk["K_neg"]; cK = _cfg(Kn)

        def _converged_step(g):
            g = g.sort_values("step").reset_index(drop=True)
            ok = ((g.acc_clean >= ACC_FLOOR) & (g.attn_dep >= DC_MIN) & (g.hidden_dep <= DH_MAX)).values
            for i in range(len(ok) - 1):
                if ok[i] and ok[i + 1]:
                    return int(g.step[i])
            return None

        st = Store("pilot_k")
        for sd in SEEDS_END:
            spec = {"block": "pilot_k", "load": Lp, "n_symbols": Kn, "seed": sd, "steps": PILOT,
                    "log": LOG, "margin": MARGIN}

            def _go(sd=sd, spec=spec):
                task = make_task("majority", cfg=cK)
                m, tr = train(task, Lp, lam=0.0, seed=sd, steps=PILOT, schedule="constant",
                              log_every=LOG, cfg=cK, ckpt_key=keyof(spec))
                return {"trace": tr.to_dict("records")}

            cached(st, spec, _go, label=f"pilot K={Kn} seed {sd}")
        if "switch" not in bk:
            PT = st.traces()
            steps_ = {sd: _converged_step(PT[PT.seed == sd]) for sd in SEEDS_END}
            good = [s for s in steps_.values() if s is not None]
            sw = max(SW_MIN, (max(good) if good else PILOT) + LOG)
            bk.update(switch=int(math.ceil(sw / LOG) * LOG),
                      pilot_not_converged=[sd for sd, s in steps_.items() if s is None])
            sel["boundary_k"] = bk; save_sel(sel)
            print(f"  K={Kn}: converged step per seed {steps_} -> switch at {bk['switch']}")

        ARM_PRICES = lambda lam: {"never": (0.0, 0.0), "late": (0.0, lam),
                                  "maintained": (lam, lam), "withdrawn": (lam, 0.0)}
        st = Store("four_arm_k"); SW = bk["switch"]; ST = SW + POST
        for arm, sd in itertools.product(NEG_ARMS, SEEDS_END):
            lb, la = ARM_PRICES(lam)[arm]
            spec = {"block": "four_arm_k", "arm": arm, "load": Lp, "n_symbols": Kn, "seed": sd,
                    "lam": lam, "switch": SW, "steps": ST, "log": LOG, "margin": MARGIN}

            def _go(arm=arm, sd=sd, lb=lb, la=la, spec=spec):
                task = make_task("majority", cfg=cK)
                m, tr = train(task, Lp, lam=lb, lam_after=la, switch=SW, seed=sd, steps=ST,
                              schedule="constant", log_every=LOG, cfg=cK, ckpt_key=keyof(spec))
                save_model(f"fourarm_{arm}_L{Lp}_K{Kn}_s{sd}", m)
                pre, post = tr[tr.step <= SW].iloc[-1], tr[tr.step > SW]
                out = {"acc_at_switch": float(pre.acc_clean), "Dh_at_switch": float(pre.hidden_dep),
                       "Dc_at_switch": float(pre.attn_dep),
                       "min_acc_post": float(post.acc_clean.min()) if len(post) else np.nan,
                       **route_eval(m, task, Lp)}
                out["trace"] = tr.to_dict("records")
                return out

            cached(st, spec, _go, label=f"K={Kn:<3} {arm:<10} seed {sd}")
        FK = st.frame(); FK.to_csv(csv("four_arm_k"), index=False)
        print(f"\nBOUNDARY CONTROL at L={Lp}, K={Kn} (lambda {lam:.4g}, switch {SW}, chance {1/Kn:.3f})")
        for arm in NEG_ARMS:
            g = FK[FK.arm == arm]
            if len(g):
                print(f"  {arm:<10} acc {fmt(g,'acc_clean')}  Dh {fmt(g,'hidden_dep',2)}  "
                      f"Dc {fmt(g,'attn_dep',2)}  inversion {int((g.hidden_dep > g.attn_dep).sum())}"
                      f"/{len(g)}  retrieval deleted {fmt(g,'acc_no_ret')}  "
                      f"acc@switch min {g.acc_at_switch.min():.3f}")


if RUN["boundary_alphabet"] and load_sel().get("lam") is not None:
    boundary_alphabet_axis()

## Section 7 — Paper tables, figure and summary

Built only from stored results. Tables are written as CSV and LaTeX to `results_majority/tables/`, and the figure to `figures_majority/fig_majority.pdf`.

In [ ]:
# Paper tables (Appendix R), built ONLY from stored results -> RESDIR/tables/*.csv|.tex
def _pm(s, d=3):
    s = pd.Series(s).dropna()
    return f"${s.mean():.{d}f}\\pm{(s.std() if len(s) > 1 else 0):.{d}f}$"


def _tex(df, path, col_fmt=None):
    with open(path, "w") as f:
        f.write(df.to_latex(index=False, escape=False, column_format=col_fmt))


def build_tables(verbose=True):
    sel = load_sel(); out = {}
    Lp, lam = sel.get("L_pos"), sel.get("lam")

    # R1: screen over all screened lengths (pre-declared grid + extension)
    if os.path.exists(csv("screen")):
        s = pd.read_csv(csv("screen")); rows = []
        for L in sorted(s.load.unique()):
            g = s[s.load == L]; h = g[g.arm == "hybrid_free"]
            rows.append({"$L$": int(L) if L in LOAD_GRID else f"{int(L)}$^\\dagger$",
                         "recurrence alone": f"{g[g.arm == 'recurrent_only_long'].acc_clean.mean():.3f}",
                         "free hybrid": f"{h.acc_clean.mean():.3f}",
                         "$D_h$": f"{h.hidden_dep.mean():.2f}", "$D_c$": f"{h.attn_dep.mean():.2f}",
                         "deletion loss": f"{h.del_ret.mean():.2f}",
                         "eff.\\ positions": f"{h.attn_eff_pos.mean():.1f}"})
        out["R1_screen"] = pd.DataFrame(rows)

    # R2: four-arm endpoints at L_pos
    if os.path.exists(csv("four_arm")) and Lp is not None:
        fa = pd.read_csv(csv("four_arm")); fa = fa[(fa.load == Lp) & np.isclose(fa.lam, lam)]; rows = []
        for arm in ["never", "late", "maintained", "withdrawn"]:
            g = fa[fa.arm == arm]
            if not len(g):
                continue
            rows.append({"arm": arm, "acc.": _pm(g.acc_clean), "$D_h$": _pm(g.hidden_dep, 2),
                         "$D_c$": _pm(g.attn_dep, 2), "ret.\\ share": _pm(g.context_share),
                         "inversion": f"{int((g.hidden_dep > g.attn_dep).sum())}/{len(g)}",
                         "complete": f"{int(((g.hidden_dep >= .9) & (g.attn_dep <= .1)).sum())}/{len(g)}",
                         "deleted": _pm(g.acc_no_ret), "ret.\\ alone": _pm(g.acc_no_rec),
                         "min": f"{g.min_acc_post.min():.3f}"})
        out["R2_four_arm"] = pd.DataFrame(rows)

    # R3: matched attenuation at non-zero shares
    if os.path.exists(csv("posthoc_curve")):
        c = pd.read_csv(csv("posthoc_curve"))
        out["R3_attenuation"] = pd.DataFrame(
            [{"retrieval share": f"{g.share.mean():.3f}",
              "free model, scaled post hoc": _pm(g.scaled_acc),
              "trained with price": f"{g.trained_acc.mean():.3f}"}
             for _, g in sorted(c.groupby("lam_ref"), key=lambda kv: -kv[1].share.mean())])

    # R4: attention structure
    if os.path.exists(csv("structure")):
        st = pd.read_csv(csv("structure")); rows = []
        for _, r in st.iterrows():
            tgt = r.get("attn_key_mass") if pd.notna(r.get("attn_key_mass")) else r.get("attn_mode_enrichment")
            rows.append({"model": r["model"], "$L$": int(r["items"]),
                         "eff.\\ positions": f"{r.attn_eff_pos:.2f}", "top-1": f"{r.attn_top1_mass:.3f}",
                         "target mass / enrichment": f"{tgt:.3f}",
                         "item mass": f"{r.attn_item_mass:.3f}",
                         "mass on $h_{T-1}$": f"{r.attn_last_kv_mass:.3f}"})
        out["R4_structure"] = pd.DataFrame(rows)

    # R5: length shift (pooled runner-up share)
    if os.path.exists(csv("fingerprint")):
        f = pd.read_csv(csv("fingerprint")); rows = []
        Ls = sorted(f.eval_load.unique())
        for arm, lab in [("never", "retrieval (never)"), ("late", "recurrence (late)"),
                         ("maintained", "recurrence (maintained)")]:
            g = f[f.arm == arm]; row = {"configuration": lab}
            for LL in Ls:
                row[f"$L{{=}}{LL}$"] = _pm(g[g.eval_load == LL].err_rate)
            last = g[g.eval_load == Ls[-1]]; n = int(last.n_err.sum())
            share = (last.runner_up_rate * last.n_err).sum() / max(n, 1)
            row[f"runner-up share at {Ls[-1]}"] = f"{share:.2f} ($n{{=}}{n}$)"
            rows.append(row)
        out["R5_length_shift"] = pd.DataFrame(rows)

    # R6: alphabet-size screen (K = N_SYMBOLS row taken from the main screen at L_pos)
    if os.path.exists(csv("screen_k")) and Lp is not None:
        k = pd.read_csv(csv("screen_k")); s = pd.read_csv(csv("screen")); rows = []
        base = s[s.load == Lp]; frames = [(N_SYMBOLS, base)] + [(K, k[k.n_symbols == K]) for K in sorted(k.n_symbols.unique())]
        for K, g in frames:
            h = g[g.arm == "hybrid_free"]
            if not len(h):
                continue
            rows.append({"$K$": int(K), "chance": f"{1 / K:.3f}",
                         "recurrence alone": f"{g[g.arm == 'recurrent_only_long'].acc_clean.mean():.3f}",
                         "free hybrid": f"{h.acc_clean.mean():.3f}",
                         "$D_h$": f"{h.hidden_dep.mean():.2f}", "$D_c$": f"{h.attn_dep.mean():.2f}",
                         "deletion loss": f"{h.del_ret.mean():.2f}"})
        out["R6_alphabet"] = pd.DataFrame(rows)

    for name, df in out.items():
        df.to_csv(TABDIR / f"{name}.csv", index=False)
        _tex(df, TABDIR / f"{name}.tex")
        if verbose:
            print(f"\n{name}\n{df.to_string(index=False)}")
    if verbose:
        print(f"\ntables -> {TABDIR}")
    return out


if RUN["tables"]:
    TABLES = build_tables()

In [ ]:
# Figure R1 (seven panels), built only from stored results.
def make_figure_r1(out_dir=FIGDIR):
    sel = load_sel(); Lp, lam, sw = sel["L_pos"], sel["lam"], sel["switch"][str(sel["L_pos"])]
    BLUE, ORANGE, GREEN, GREY = "#1b6ca8", "#e07a1f", "#2a9d8f", "#8d99ae"
    mpl.rcParams.update({"font.size": 6.5, "axes.titlesize": 7, "legend.fontsize": 5.4,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "pdf.fonttype": 42})
    traces = Store("four_arm").traces()
    traces = traces[(traces.load == Lp) & (traces.lam == lam)]
    screen = pd.read_csv(csv("screen")); screen = screen[screen.load.isin(LOAD_GRID)]
    struct = pd.read_csv(csv("structure")); price = pd.read_csv(csv("price"))
    curve = pd.read_csv(csv("posthoc_curve")) if os.path.exists(csv("posthoc_curve")) else None

    fig = plt.figure(figsize=(5.5, 3.9))
    gs = fig.add_gridspec(2, 12, hspace=0.7, wspace=4.2)
    cols = ["hidden_dep", "attn_dep", "acc_clean"]
    for j, arm in enumerate(["never", "late", "maintained", "withdrawn"]):
        ax = fig.add_subplot(gs[0, 3 * j:3 * j + 3])
        g = traces[traces.arm == arm].groupby("step")
        mu, sd = g[cols].mean(), g[cols].std().fillna(0)
        x = mu.index.values - sw
        for c, color, label in [("hidden_dep", BLUE, "$D_h$"), ("attn_dep", ORANGE, "$D_c$")]:
            ax.plot(x, mu[c], color=color, lw=1.2, label=label)
            ax.fill_between(x, mu[c] - sd[c], mu[c] + sd[c], color=color, alpha=0.18, lw=0)
        ax.plot(x, mu.acc_clean, color=GREEN, ls="--", lw=1, label="accuracy")
        ax.axvline(0, color="k", lw=0.6, ls=":"); ax.set_ylim(-0.03, 1.05)
        ax.set_title(f"({'abcd'[j]}) {arm}"); ax.set_xlabel("steps after switch")
        if j == 0:
            ax.set_ylabel("dependence / accuracy")
        else:
            ax.set_yticklabels([])
        if j == 3:
            ax.legend(frameon=False, loc="center right")

    loads = sorted(screen.load.unique())
    hybrid = screen[screen.arm == "hybrid_free"].groupby("load")
    ax = fig.add_subplot(gs[1, 0:4])
    rec = screen[screen.arm == "recurrent_only_long"].groupby("load").acc_clean.mean()
    ax.plot(loads, rec.values, "o-", color=BLUE, ms=3.5, lw=1.2, label="rec. alone")
    ax.plot(loads, hybrid.acc_clean.mean().values, "s--", color=ORANGE, ms=3, lw=1,
            mfc="none", label="free hybrid")
    ax.errorbar(loads, hybrid.acc_no_ret.mean().values, hybrid.acc_no_ret.std().values,
                fmt="o-", color=GREY, ms=3, lw=1, capsize=1.5, label="retr. deleted")
    ax.axhline(1 / N_SYMBOLS, color="k", lw=0.5, ls=":"); ax.text(4.1, 0.145, "chance", fontsize=5.5)
    ax.set_xscale("log", base=2); ax.set_xticks(loads); ax.set_xticklabels(loads)
    ax.set_ylim(0, 1.06); ax.set_xlabel("items $L$"); ax.set_ylabel("accuracy")
    ax.set_title("(e) viability, necessity"); ax.legend(frameon=False, loc=(0.36, 0.56), handlelength=1.5)

    ax = fig.add_subplot(gs[1, 4:8]); eff = hybrid.attn_eff_pos
    ax.errorbar(loads, eff.mean().values, eff.std().values, fmt="o-", color=ORANGE, ms=3.5, lw=1.2)
    ax.plot(loads, [n + CFG.base_delay for n in loads], color="k", lw=0.6, ls=":")
    sr = struct[struct.model.str.startswith("selective")].iloc[0]
    ax.plot([sr["items"]], [sr.attn_eff_pos], "D", color=GREY, ms=4.5)
    ax.set_xscale("log", base=2); ax.set_yscale("log", base=2)
    ax.set_xticks(loads); ax.set_xticklabels(loads)
    ax.set_yticks([1, 2, 4, 8, 16, 32]); ax.set_yticklabels([1, 2, 4, 8, 16, 32])
    ax.set_xlabel("items $L$"); ax.set_ylabel("eff. positions", labelpad=1)
    ax.set_title("(f) pooling, not lookup")
    ax.text(10.5, 6.2, "majority", color=ORANGE, fontsize=5.6)
    ax.text(4.3, 9.5, "uniform over K/V", fontsize=5.4, rotation=19)
    ax.text(18.5, 2.3, "selective\nrecall", color=GREY, fontsize=5.6, ha="center")

    ax = fig.add_subplot(gs[1, 8:12])
    if curve is not None and len(curve):
        g = curve.groupby("lam_ref"); shares = g.share.mean(); sc = g.scaled_acc.agg(["mean", "std"])
        tr = price.groupby("lam")[["context_share", "acc_clean"]].mean().loc[shares.index]
        ax.errorbar(shares.values, sc["mean"].values, sc["std"].values, fmt="o-", color=GREY,
                    ms=3.5, lw=1.2, capsize=1.5, label="free, scaled")
        ax.plot(tr.context_share.values, tr.acc_clean.values, "s-", color=BLUE, ms=3.5, lw=1.2,
                label="trained with price")
        ax.legend(frameon=False, loc=(0.36, 0.1), handlelength=1.5)
    ax.axhline(1 / N_SYMBOLS, color="k", lw=0.5, ls=":")
    ax.set_xlim(-0.03, 0.83); ax.set_ylim(0, 1.06)
    ax.set_xlabel("retrieval contribution share"); ax.set_ylabel("accuracy")
    ax.set_title("(g) matched attenuation")
    for ext in ("pdf", "png"):
        fig.savefig(pathlib.Path(out_dir) / f"fig_majority.{ext}", dpi=300, bbox_inches="tight")
    return fig


if RUN["figure"] and load_sel().get("lam") is not None:
    make_figure_r1(); plt.show()
    print(f"figure -> {FIGDIR / 'fig_majority.pdf'}")

In [ ]:
if RUN["summary"]:
    sel = load_sel(); Lp, lam = sel.get("L_pos"), sel.get("lam")
    fa = pd.read_csv(csv("four_arm")) if os.path.exists(csv("four_arm")) else pd.DataFrame()
    fa = fa[(fa.load == Lp) & np.isclose(fa.lam, lam)] if len(fa) else fa
    late, never = (fa[fa.arm == "late"], fa[fa.arm == "never"]) if len(fa) else (fa, fa)
    curve = pd.read_csv(csv("posthoc_curve")) if os.path.exists(csv("posthoc_curve")) else None
    chk = {}
    if len(late) and len(never):
        chk["late arm inverts (Dh > Dc) in every seed"] = bool((late.hidden_dep > late.attn_dep).all())
        chk["late accuracy preserved (>= VIABLE_ACC)"] = bool((late.acc_clean >= VIABLE_ACC).all())
        chk["never arm stays retrieval-dominant"] = bool((never.hidden_dep <= never.attn_dep).all())
        chk["retrieval deletable after handover"] = bool(((late.acc_clean - late.acc_no_ret) <= DEL_SAFE).all())
        chk["retrieval load-bearing in never arm"] = bool((never.del_ret >= DELETION_MIN).all())
    if curve is not None and len(curve):
        mid = curve[curve.share.between(0.05, 0.95)]
        chk["post-hoc scaling at non-zero shares fails (mean gap >= 0.1)"] = bool(
            (mid.trained_acc - mid.scaled_acc).mean() >= 0.1)
    for k, v in chk.items():
        print(f"  [{'x' if v else ' '}] {k}")

    L_neg, bk = sel.get("L_neg"), sel.get("boundary_k")
    if L_neg is not None:
        fa_all = Store("four_arm").frame()
        ln = fa_all[(fa_all.load == L_neg) & (fa_all.arm == "late")] if len(fa_all) else fa_all
        boundary = (f"length control L={L_neg}: late inverts in "
                    f"{int((ln.hidden_dep > ln.attn_dep).sum())}/{len(ln)}")
    elif bk:
        boundary = f"alphabet control K={bk.get('K_neg')} (see Section 6b output)"
    else:
        boundary = ("none — recurrence viable at every screened length and pooled retrieval "
                    "degrades first on the alphabet axis; Majority is positive-only (not a Table 2 row)")
    print(f"\n  boundary control: {boundary}")
    if chk and all(chk.values()):
        print("\nVERDICT: positive handover SUPPORTED" +
              ("" if (L_neg is not None or bk) else " (positive side only)"))
    elif chk:
        print("\nVERDICT: positive handover NOT fully supported — see unchecked items")
    else:
        print("\nVERDICT: incomplete — four-arm results not found")